# False positives: CRG vs module (wrong rows)
Wrong row = a row the tool reports that the language's compiler reference does not have (reported − matched).
Same 37 languages, same reference, newest runs: `checks.csv` (CRG) and `module-checks.csv` (module). Only kinds both sides measured (N/A kinds left out). cpp and objc have no module run, so they're left out.
Output: `results/language-survey/fp-crg-vs-module.csv`.

In [1]:
import survey_lab as L, pandas as pd
c = L.latest(); m = L.latest_module()
key = ['language','kind']
c = c[c.CRG.notna() & c.expected.notna()]; m = m[m.CRG.notna() & m.expected.notna()]
j = c[key+['CRG','matched']].merge(m[key+['CRG','matched']], on=key, suffixes=('_crg','_mod'))
j['crg_wrong'] = (j.CRG_crg - j.matched_crg).astype(int); j['crg_rep'] = j.CRG_crg.astype(int)
j['mod_wrong'] = (j.CRG_mod - j.matched_mod).astype(int); j['mod_rep'] = j.CRG_mod.astype(int)
print(j.language.nunique(), 'languages')

37 languages


In [2]:
tot = j.groupby('kind')[['crg_wrong','crg_rep','mod_wrong','mod_rep']].sum()
tot.loc['ALL'] = tot.sum()
tot['crg %'] = (100*tot.crg_wrong/tot.crg_rep).round(1); tot['mod %'] = (100*tot.mod_wrong/tot.mod_rep).round(1)
tot.sort_values('crg %', ascending=False)

,crg_wrong,crg_rep,mod_wrong,mod_rep,crg %,mod %
kind,,,,,,
FUNCTIONS,52304,148041,2491,104625,35.3,2.4
CONTAINS,53455,162258,2510,118264,32.9,2.1
ALL,165250,1044049,9043,950716,15.8,1.0
CLASSES,2330,15604,19,13639,14.9,0.1
CALLS,54113,673177,3774,669676,8.0,0.6
IMPORTS_FROM,2672,36890,218,35906,7.2,0.6
INHERITS,376,8079,31,8606,4.7,0.4


In [3]:
j['crg %'] = (100*j.crg_wrong/j.crg_rep.where(j.crg_rep>0)).round(1)
j['mod %'] = (100*j.mod_wrong/j.mod_rep.where(j.mod_rep>0)).round(1)
out = j[['language','kind','crg_wrong','crg_rep','crg %','mod_wrong','mod_rep','mod %']].sort_values(['crg_wrong'], ascending=False)
out.to_csv('../results/language-survey/fp-crg-vs-module.csv', index=False)
tot.reset_index().to_csv('../results/language-survey/fp-crg-vs-module-totals.csv', index=False)
print('module above 5% wrong:', (out['mod %']>5).sum(), 'of', len(out))
out[out['mod %']>5]

module above 5% wrong: 16 of 196


,language,kind,crg_wrong,crg_rep,crg %,mod_wrong,mod_rep,mod %
165,typescript,CONTAINS,3945,7055,55.9,216,4201,5.1
8,c,CALLS,3481,9321,37.3,1292,6984,18.5
163,typescript,FUNCTIONS,2963,5264,56.3,216,3391,6.4
113,rescript,FUNCTIONS,1625,1868,87.0,38,299,12.7
155,swift,CONTAINS,753,2546,29.6,685,2572,26.6
6,c,CONTAINS,605,1440,42.0,131,974,13.4
5,c,CLASSES,519,556,93.3,3,48,6.2
153,swift,FUNCTIONS,452,1951,23.2,685,2278,30.1
187,xslt,FUNCTIONS,413,5373,7.7,411,5371,7.7
188,xslt,CONTAINS,413,5373,7.7,411,5371,7.7


## Summary numbers and repo list (for the sheet's Summary tab)

In [4]:
c2 = L.latest(); m2 = L.latest_module()
c2 = c2[c2.expected.notna() & c2.CRG.notna()]; m2 = m2[m2.expected.notna() & m2.CRG.notna()]
k = ['language','kind']; both = c2[k+['expected','matched']].merge(m2[k+['expected','matched']], on=k, suffixes=('_crg','_mod'))
found = dict(crg_found=int(both.matched_crg.sum()), crg_exp=int(both.expected_crg.sum()), mod_found=int(both.matched_mod.sum()), mod_exp=int(both.expected_mod.sum()))
print(found, round(100*found['crg_found']/found['crg_exp'],1), round(100*found['mod_found']/found['mod_exp'],1))
import csv
R = L.repos(); third = {r['language']: r['slug'] for r in csv.DictReader(open('../thirdset-lab/thirdset_repos.csv'))}
repo_list = pd.DataFrame([dict(stack=l, survey_repo=R[l]['folder'].split('__',1)[1], thirdset_repo=third.get(l,'').split('__',1)[-1],
                               crg_checked=l in set(L.latest().language), module_checked=l in set(L.latest_module().language)) for l in sorted(R)])
repo_list.to_csv('../results/language-survey/repo-list.csv', index=False)
pd.Series(found).to_csv('../results/language-survey/found-totals.csv')
repo_list

{'crg_found': 878799, 'crg_exp': 947202, 'mod_found': 941673, 'mod_exp': 947202} 92.8 99.4


,stack,survey_repo,thirdset_repo,crg_checked,module_checked
0,bash,bash-it-bash-it,bats-core-bats-core,True,True
1,c,jqlang-jq,redis-redis,True,True
2,cobol,meyfa-cobolcraft,,False,False
3,cpp,gabime-spdlog,nlohmann-json,True,False
4,csharp,jamesnk-newtonsoft.json,luckypennysoftware-automapper,True,True
5,dart,felangel-bloc,cfug-dio,True,True
6,elixir,phoenixframework-phoenix,plausible-analytics,True,True
7,gdscript,orama-interactive-pixelorama,godotengine-godot-demo-projects,True,True
8,go,caddyserver-caddy,junegunn-fzf,True,True
9,hcl,terraform-aws-modules-terraform-aws-eks,terraform-aws-modules-terraform-aws-vpc,True,True
